In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
kagglehub.competition_download(
    'kmu-rec-sys-26-rating-prediction',
    output_dir = "dataset"
)

In [ ]:
!head dataset/train_small.csv

In [ ]:
import csv
import torch
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

userIds, itemIds = {}, {}
n_users, n_items, n_ratings = 0,0,0
users, items, ratings = [], [], []

with open('dataset/train_small.csv',"r") as f:
  reader = csv.reader(f)
  next(reader)
  for userId, itemId, rating, _ in reader:
    if userId not in userIds:
      userIds[userId] = n_users
      n_users+=1
    if itemId not in itemIds:
      itemIds[itemId] = n_items
      n_items+=1

    users.append(userIds[userId])
    items.append(itemIds[itemId])
    ratings.append(float(rating))

users = torch.tensor(users).to(device)
items = torch.tensor(items).to(device)
ratings = torch.tensor(ratings).to(device)

In [ ]:
userIds

In [ ]:
n_train = int(len(ratings)*0.9)
indices = torch.randperm(len(ratings)) # 데이터 섞기

train_indices = indices[:n_train]
valid_indices = indices[n_train:]

users_valid = users[valid_indices]
items_valid = items[valid_indices]
ratings_valid = ratings[valid_indices]

users = users[train_indices]
items = items[train_indices]
ratings = ratings[train_indices]

users_all = users.clone()
items_all = items.clone()
ratings_all = ratings.clone()

In [ ]:
lmd = 15
rank = 10 # 벡터의 차원
lr = 0.01
# mean + user_bias + item_bias + user_emb*item_emb
mean = ratings.mean().to(device)

user_bias = torch.zeros(n_users, requires_grad=True, device=device)
item_bias = torch.zeros(n_items, requires_grad = True, device=device)

user_emb = torch.normal(mean=0, std=0.01, size=(n_users,rank), requires_grad=True, device=device)
item_emb = torch.normal(mean=0, std=0.01, size=(n_items,rank), requires_grad=True, device=device)
optim = torch.optim.Adam([user_bias, item_bias, user_emb, item_emb],lr=lr)



In [ ]:
a = torch.tensor([9,8,7,6,5])

idx = torch.tensor([0,2,2,1,3,4])

a[idx]

In [ ]:
best_val = float('inf')
best_epoch = 0

for epoch in range(1000):

    h = mean + user_bias[users] + item_bias[items] + \
    torch.sum(user_emb[users] * item_emb[items], dim=1)

    sse = ((h - ratings)**2).sum()
    reg = lmd * ((user_bias**2).sum() + (item_bias**2).sum() +
                 (user_emb**2).sum() + (item_emb**2).sum())

    cost = sse + reg

    optim.zero_grad()
    cost.backward()
    optim.step()

    # validation
    if epoch % 50 == 0:
        with torch.no_grad():
            h_val = mean + user_bias[users_valid] + item_bias[items_valid] + \
                    torch.sum(user_emb[users_valid] * item_emb[items_valid], dim=1)

            mse_val = ((h_val - ratings_valid)**2).mean().item()

            print(f"epoch:{epoch}, val:{mse_val:.4f}")

            if mse_val < best_val:
                best_val = mse_val
                best_epoch = epoch

print(f"BEST EPOCH: {best_epoch}")

In [ ]:
# 전체 데이터로 다시 학습

mean = ratings_all.mean()

user_bias = torch.nn.Parameter(torch.zeros(n_users, device=device))
item_bias = torch.nn.Parameter(torch.zeros(n_items, device=device))

user_emb = torch.nn.Parameter(torch.randn(n_users, rank, device=device) * 0.01)
item_emb = torch.nn.Parameter(torch.randn(n_items, rank, device=device) * 0.01)

optim = torch.optim.Adam([user_bias, item_bias, user_emb, item_emb], lr=lr)

# best_epoch 만큼 학습
for epoch in range(best_epoch):

    h = mean + user_bias[users_all] + item_bias[items_all] + \
        torch.sum(user_emb[users_all] * item_emb[items_all], dim=1)

    sse = ((h - ratings_all)**2).sum()
    reg = lmd * ((user_bias**2).sum() + (item_bias**2).sum() +
                 (user_emb**2).sum() + (item_emb**2).sum())

    cost = sse + reg

    optim.zero_grad()
    cost.backward()
    optim.step()

    if epoch % 100 == 0:
        mse = (sse / len(ratings_all)).item()
        print(f"epoch:{epoch}, mse:{mse:.4f}")

In [ ]:
with open('dataset/test_small.csv') as fin, \
     open('solution.csv','w', newline='') as fout:

     reader = csv.reader(fin)
     next(reader)

     writer = csv.writer(fout)
     writer.writerow(["id", "rating"])
     for userId, itemId in reader:
      uid = userIds[userId]
      iid = itemIds[itemId]
      pred = mean + user_bias[uid] + item_bias[iid] + \
      (user_emb[uid]*item_emb[iid]).sum()
      pred = min(5, max(0.5, pred.item()))
      writer.writerow([f"{userId}_{itemId}", pred])
